# Reproducibility note

The final test reported in the README was produced with the Colab notebook using the same fixed hyperparameters and TRAIN+VALID refit. This published notebook has execution output and personal Drive paths removed. Running it requires the user-provided split parquet files and may take substantial GPU time. Do not use TEST results for further tuning.


# M5: Final One-Time TEST Evaluation

This notebook performs the **final refit and one-time TEST evaluation** for the selected architecture.

Frozen architecture and hyperparameters:

`BPR top-500 + fine-tuned recency-aware MiniLM metadata top-500 -> union -> 50/50 normalized linear reranking -> top-10`

Fixed settings:
- BPR: 64-d embeddings, Adam `lr=1e-3`, `weight_decay=1e-6`, batch `2048`, `10` epochs
- MiniLM: `all-MiniLM-L6-v2`, contrastive fine-tuning with `MultipleNegativesRankingLoss`
- MiniLM fine-tuning: up to `60,000` co-interaction pairs, batch `64`, `1` epoch, `lr=2e-5`, max length `256`
- Candidate K per retriever: `500`
- Metadata recent window: latest `2` interactions
- Recency weight: `rho=0.60`
- Final fusion: `0.50 * BPR_z + 0.50 * Metadata_z`

## Final-evaluation protocol

1. **TRAIN + VALID are combined** as final training evidence.
2. BPR is retrained from scratch on TRAIN + VALID using the already-selected M1a recipe.
3. MiniLM is refit from the original pretrained checkpoint on TRAIN + VALID using the already-selected contrastive recipe.
4. Metadata user profiles are rebuilt from TRAIN + VALID.
5. TEST is evaluated exactly once with the frozen retrieval/reranking hyperparameters.

**Do not use the resulting TEST metrics to tune this model further.**


## 1. Imports and fixed configuration


In [ ]:
# Optional for Google Colab: mount Drive, then set DATA_DIR in the next cell.
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import gc
import json
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.sparse import csr_matrix
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from datasets import Dataset as HFDataset
    from sentence_transformers import (
        SentenceTransformer,
        SentenceTransformerTrainer,
        SentenceTransformerTrainingArguments,
        losses,
    )
    from sentence_transformers.sentence_transformer.training_args import BatchSamplers
except ImportError as exc:
    raise ImportError(
        "Final MiniLM refit requires sentence-transformers, datasets and accelerate. "
        "Install with: %pip install -U sentence-transformers datasets 'accelerate>=1.1.0'"
    ) from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = Path(".")

TRAIN_PATH = DATA_DIR / 'train_interactions.parquet'
VALID_PATH = DATA_DIR / 'valid_interactions.parquet'
TEST_PATH = DATA_DIR / 'test_interactions.parquet'
METADATA_PATH = DATA_DIR / 'product_metadata.parquet'
M2_INDEX_PATH = DATA_DIR / 'metadata_embedding_index.parquet'

FINAL_BPR_PATH = DATA_DIR / 'bpr_final_train_valid.pt'
FINAL_MINILM_DIR = DATA_DIR / 'minilm_recommender_finetuned_final'
FINAL_META_EMBED_PATH = DATA_DIR / 'metadata_minilm_finetuned_final_embeddings.npy'
FINAL_RESULTS_PATH = DATA_DIR / 'final_test_results.json'

# Frozen BPR settings
BPR_DIM = 64
BPR_BATCH_SIZE = 2048
BPR_EPOCHS = 10
BPR_LR = 1e-3
BPR_WEIGHT_DECAY = 1e-6

# Frozen MiniLM fine-tuning settings
BASE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
MAX_TRAIN_PAIRS = 60_000
TRAIN_BATCH_SIZE = 64
NUM_EPOCHS = 1
LEARNING_RATE = 2e-5
WARMUP_FRACTION = 0.10
WEIGHT_DECAY = 0.01
MAX_SEQ_LENGTH = 256
ENCODE_BATCH_SIZE = 128

# Frozen retrieval/reranking settings
CANDIDATE_K = 500
RECENT_N = 2
RHO = 0.60
ALPHA_BPR = 0.50
ALPHA_META = 0.50
EVAL_BATCH_SIZE = 96

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))
print('This notebook will refit BPR + MiniLM on TRAIN+VALID, then evaluate TEST once.')


## 2. Load TRAIN / VALID / TEST and lightweight metadata columns


In [ ]:
train = pd.read_parquet(TRAIN_PATH)
valid = pd.read_parquet(VALID_PATH)
test = pd.read_parquet(TEST_PATH)

# Load only fields required to reconstruct the exact M2 text.
metadata = pd.read_parquet(
    METADATA_PATH,
    columns=['parent_asin', 'title', 'categories', 'features', 'description'],
).reset_index(drop=True)
metadata_index = pd.read_parquet(M2_INDEX_PATH)

assert len(metadata) == len(metadata_index)
assert np.array_equal(
    metadata['parent_asin'].astype(str).to_numpy(),
    metadata_index['parent_asin'].astype(str).to_numpy(),
), 'Metadata row order differs from metadata_embedding_index.parquet.'

# VALID is now allowed as training evidence because architecture/hyperparameters are frozen.
final_train = pd.concat([train, valid], ignore_index=True)

print('TRAIN       :', train.shape)
print('VALID       :', valid.shape)
print('TRAIN+VALID :', final_train.shape)
print('TEST        :', test.shape)
print('Metadata    :', metadata.shape)

assert test['user_id'].nunique() == len(test), 'Expected one TEST target per user.'


## 3. Reconstruct the established global ID mapping

For consistency with M1/M4, preserve first-appearance ordering:

- users: `TRAIN.user_id.unique()`
- items: `TRAIN -> VALID -> TEST`, then `.unique()`

TEST interactions are **not** used as training labels. TEST item IDs are present only to define the benchmark catalog/global mapping.


In [ ]:
all_users = train['user_id'].unique()
all_items = pd.concat([
    train['parent_asin'],
    valid['parent_asin'],
    test['parent_asin'],
]).unique()

user_to_idx = {user: idx for idx, user in enumerate(all_users)}
item_to_idx = {item: idx for idx, item in enumerate(all_items)}
idx_to_item = np.asarray(all_items)

NUM_USERS = len(all_users)
NUM_ITEMS = len(all_items)

assert final_train['user_id'].isin(user_to_idx).all()
assert test['user_id'].isin(user_to_idx).all()

print('Users:', NUM_USERS)
print('Items:', NUM_ITEMS)


## 4. Index TRAIN+VALID and build final seen-item sets


In [ ]:
final_train_indexed = final_train[['user_id', 'parent_asin', 'timestamp']].copy()
final_train_indexed['user_idx'] = (
    final_train_indexed['user_id'].map(user_to_idx).astype(np.int32)
)
final_train_indexed['item_idx'] = (
    final_train_indexed['parent_asin'].map(item_to_idx).astype(np.int32)
)

final_user_seen = (
    final_train_indexed
    .groupby('user_idx')['item_idx']
    .apply(set)
    .to_dict()
)

train_pairs = final_train_indexed[['user_idx', 'item_idx']].to_numpy(dtype=np.int32)

# Only items observed positively in TRAIN+VALID are eligible for BPR retrieval.
final_bpr_catalog_idx = np.array(
    sorted({item_to_idx[item] for item in final_train['parent_asin'].unique()}),
    dtype=np.int32,
)

global_to_bpr_pos = np.full(NUM_ITEMS, -1, dtype=np.int32)
global_to_bpr_pos[final_bpr_catalog_idx] = np.arange(
    len(final_bpr_catalog_idx), dtype=np.int32
)

bpr_trained_mask = np.zeros(NUM_ITEMS, dtype=bool)
bpr_trained_mask[final_bpr_catalog_idx] = True

print('Final training pairs :', len(train_pairs))
print('BPR retrieval catalog:', len(final_bpr_catalog_idx))


## 5. Final BPR refit on TRAIN+VALID


In [ ]:
class BPRDataset(Dataset):
    def __init__(self, train_pairs, user_seen_items, num_items):
        self.train_pairs = train_pairs
        self.user_seen_items = user_seen_items
        self.num_items = num_items

    def __len__(self):
        return len(self.train_pairs)

    def __getitem__(self, idx):
        user_idx, positive_item_idx = self.train_pairs[idx]
        user_idx = int(user_idx)
        positive_item_idx = int(positive_item_idx)
        seen = self.user_seen_items[user_idx]

        while True:
            negative_item_idx = random.randrange(self.num_items)
            if negative_item_idx not in seen:
                break

        return (
            torch.tensor(user_idx, dtype=torch.long),
            torch.tensor(positive_item_idx, dtype=torch.long),
            torch.tensor(negative_item_idx, dtype=torch.long),
        )


class BPRModel(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)

    def forward(self, user_idx, positive_item_idx, negative_item_idx):
        user_vec = self.user_embedding(user_idx)
        positive_vec = self.item_embedding(positive_item_idx)
        negative_vec = self.item_embedding(negative_item_idx)
        positive_score = (user_vec * positive_vec).sum(dim=1)
        negative_score = (user_vec * negative_vec).sum(dim=1)
        return positive_score, negative_score


def bpr_loss(positive_score, negative_score):
    return -F.logsigmoid(positive_score - negative_score).mean()


bpr_dataset = BPRDataset(
    train_pairs=train_pairs,
    user_seen_items=final_user_seen,
    num_items=NUM_ITEMS,
)

bpr_loader = DataLoader(
    bpr_dataset,
    batch_size=BPR_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

bpr_model = BPRModel(
    num_users=NUM_USERS,
    num_items=NUM_ITEMS,
    embedding_dim=BPR_DIM,
).to(DEVICE)

optimizer = torch.optim.Adam(
    bpr_model.parameters(),
    lr=BPR_LR,
    weight_decay=BPR_WEIGHT_DECAY,
)

for epoch in range(BPR_EPOCHS):
    bpr_model.train()
    total_loss = 0.0

    for users, positives, negatives in bpr_loader:
        users = users.to(DEVICE, non_blocking=True)
        positives = positives.to(DEVICE, non_blocking=True)
        negatives = negatives.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        positive_score, negative_score = bpr_model(users, positives, negatives)
        loss = bpr_loss(positive_score, negative_score)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f'Epoch {epoch + 1:02d} | Loss: {total_loss / len(bpr_loader):.4f}')

# Save only the expensive final checkpoint.
torch.save(bpr_model.state_dict(), FINAL_BPR_PATH)
print('Saved:', FINAL_BPR_PATH)

# Keep compact NumPy representations for evaluation, then free GPU model state.
bpr_user_embeddings = (
    bpr_model.user_embedding.weight.detach().cpu().numpy().astype(np.float32)
)
bpr_item_embeddings = (
    bpr_model.item_embedding.weight.detach().cpu().numpy().astype(np.float32)
)

del bpr_model, optimizer, bpr_loader, bpr_dataset
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 6. Rebuild the exact M2 metadata text

This uses the same fields selected in M2: title, categories, features, and description.


In [ ]:
def clean_field(value):
    if value is None:
        return ''
    if isinstance(value, (list, tuple, np.ndarray)):
        return ' '.join(str(x) for x in value if x is not None)
    try:
        if pd.isna(value):
            return ''
    except Exception:
        pass
    return str(value)

for col in ['title', 'categories', 'features', 'description']:
    metadata[col + '_text'] = metadata[col].apply(clean_field)

metadata['content_text'] = (
    'Title: ' + metadata['title_text']
    + '. Categories: ' + metadata['categories_text']
    + '. Features: ' + metadata['features_text']
    + '. Description: ' + metadata['description_text']
)

print('Metadata text rows:', len(metadata))
print('Example length:', len(metadata.iloc[0]['content_text']))


## 7. Build the final contrastive MiniLM dataset from TRAIN+VALID

The fine-tuning recipe is frozen from M2.2:

- choose one distinct adjacent co-interaction pair per user
- deduplicate pairs
- cap at 60,000 pairs
- randomly swap direction for half of pairs
- no TEST interactions are used


In [ ]:
rng = np.random.default_rng(SEED)

chronological = final_train[['user_id', 'parent_asin', 'timestamp']].sort_values(
    ['user_id', 'timestamp'], kind='mergesort'
)

pair_rows = []
for _, group in tqdm(
    chronological.groupby('user_id', sort=False),
    total=chronological['user_id'].nunique(),
    desc='Build final MiniLM pairs',
):
    items = group['parent_asin'].to_numpy()
    if len(items) < 2:
        continue

    valid_positions = np.where(items[:-1] != items[1:])[0]
    if len(valid_positions) == 0:
        continue

    j = int(rng.choice(valid_positions))
    pair_rows.append((items[j], items[j + 1]))

positive_pairs = pd.DataFrame(
    pair_rows,
    columns=['anchor_asin', 'positive_asin'],
).drop_duplicates().reset_index(drop=True)

if len(positive_pairs) > MAX_TRAIN_PAIRS:
    positive_pairs = positive_pairs.sample(
        n=MAX_TRAIN_PAIRS,
        random_state=SEED,
    ).reset_index(drop=True)

text_lookup = metadata.set_index('parent_asin')['content_text'].to_dict()
positive_pairs['anchor'] = positive_pairs['anchor_asin'].map(text_lookup)
positive_pairs['positive'] = positive_pairs['positive_asin'].map(text_lookup)
positive_pairs = positive_pairs.dropna(subset=['anchor', 'positive']).reset_index(drop=True)

swap_rng = np.random.default_rng(SEED + 1)
swap_mask = swap_rng.random(len(positive_pairs)) < 0.5
anchor_copy = positive_pairs.loc[swap_mask, 'anchor'].copy()
positive_pairs.loc[swap_mask, 'anchor'] = (
    positive_pairs.loc[swap_mask, 'positive'].to_numpy()
)
positive_pairs.loc[swap_mask, 'positive'] = anchor_copy.to_numpy()

train_dataset = HFDataset.from_dict({
    'anchor': positive_pairs['anchor'].tolist(),
    'positive': positive_pairs['positive'].tolist(),
})

print('Final MiniLM pairs:', len(train_dataset))
print('Unique anchor items  :', positive_pairs['anchor_asin'].nunique())
print('Unique positive items:', positive_pairs['positive_asin'].nunique())

# Free duplicated pair text before loading/training the Transformer.
del text_lookup, positive_pairs, pair_rows, chronological
gc.collect()


## 8. Final MiniLM refit on TRAIN+VALID

GPU is strongly recommended. The same selected fine-tuning recipe is used; no hyperparameter is tuned here.


In [ ]:
encoder = SentenceTransformer(BASE_MODEL, device=str(DEVICE))
encoder.max_seq_length = MAX_SEQ_LENGTH

train_loss = losses.MultipleNegativesRankingLoss(encoder)

steps_per_epoch = math.ceil(len(train_dataset) / TRAIN_BATCH_SIZE)
warmup_steps = max(1, int(round(WARMUP_FRACTION * steps_per_epoch * NUM_EPOCHS)))

training_args = SentenceTransformerTrainingArguments(
    output_dir='minilm_final_refit_tmp',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,
    weight_decay=WEIGHT_DECAY,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy='no',
    eval_strategy='no',
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
    bf16=False,
    dataloader_num_workers=2 if torch.cuda.is_available() else 0,
    dataloader_pin_memory=torch.cuda.is_available(),
    seed=SEED,
)

trainer = SentenceTransformerTrainer(
    model=encoder,
    args=training_args,
    train_dataset=train_dataset,
    loss=train_loss,
)

print('Training device:', encoder.device)
print('Warmup steps   :', warmup_steps)
trainer.train()
print('Final MiniLM refit complete.')

encoder.save(str(FINAL_MINILM_DIR))
print('Saved model:', FINAL_MINILM_DIR)


## 9. Encode the full product catalog with the final fine-tuned MiniLM


In [ ]:
final_metadata_embeddings = encoder.encode(
    metadata['content_text'].tolist(),
    batch_size=ENCODE_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float32)

np.save(FINAL_META_EMBED_PATH, final_metadata_embeddings)
print('Saved embeddings:', FINAL_META_EMBED_PATH)
print('Shape:', final_metadata_embeddings.shape)

# Transformer is no longer needed for downstream ranking.
del trainer, encoder, train_dataset
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 10. Align final metadata embeddings and build TRAIN+VALID long-term + recent-2 profiles


In [ ]:
asin_to_meta_idx = {
    asin: idx for idx, asin in enumerate(metadata_index['parent_asin'])
}

global_to_meta_idx = np.array(
    [asin_to_meta_idx.get(asin, -1) for asin in all_items],
    dtype=np.int32,
)

missing_metadata = int((global_to_meta_idx < 0).sum())
print('Global items missing metadata:', missing_metadata)
assert missing_metadata == 0

# Ensure embeddings are normalized.
final_metadata_embeddings /= np.clip(
    np.linalg.norm(final_metadata_embeddings, axis=1, keepdims=True),
    1e-12,
    None,
)

metadata_catalog_embeddings = final_metadata_embeddings[global_to_meta_idx]

final_train_indexed['meta_idx'] = (
    final_train_indexed['parent_asin'].map(asin_to_meta_idx).astype(np.int32)
)


def l2_normalize_rows(matrix):
    matrix = np.asarray(matrix, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    ok = norms[:, 0] > 1e-12
    matrix[ok] /= norms[ok]
    return matrix, ok


# Long-term profile from every TRAIN+VALID interaction.
all_history_matrix = csr_matrix(
    (
        np.ones(len(final_train_indexed), dtype=np.float32),
        (final_train_indexed['user_idx'], final_train_indexed['meta_idx']),
    ),
    shape=(NUM_USERS, len(metadata_index)),
    dtype=np.float32,
)

all_history_profiles = np.asarray(
    all_history_matrix @ final_metadata_embeddings,
    dtype=np.float32,
)
all_history_profiles, has_all_profile = l2_normalize_rows(all_history_profiles)

# Recent profile from latest 2 TRAIN+VALID interactions.
chronological_final = final_train_indexed.sort_values(
    ['user_idx', 'timestamp'], kind='mergesort'
)
recent_rows = (
    chronological_final
    .groupby('user_idx', sort=False, group_keys=False)
    .tail(RECENT_N)
)

recent_matrix = csr_matrix(
    (
        np.ones(len(recent_rows), dtype=np.float32),
        (recent_rows['user_idx'], recent_rows['meta_idx']),
    ),
    shape=(NUM_USERS, len(metadata_index)),
    dtype=np.float32,
)

recent_profiles = np.asarray(
    recent_matrix @ final_metadata_embeddings,
    dtype=np.float32,
)
recent_profiles, has_recent_profile = l2_normalize_rows(recent_profiles)

print('All-history profiles:', all_history_profiles.shape)
print('Recent profiles     :', recent_profiles.shape)


## 11. Prepare TEST users and targets


In [ ]:
test_user_idx = np.array(
    [user_to_idx[user] for user in test['user_id'].to_numpy()],
    dtype=np.int32,
)

test_target_idx = np.array(
    [item_to_idx[item] for item in test['parent_asin'].to_numpy()],
    dtype=np.int32,
)

NUM_TEST = len(test_user_idx)
cold_test_targets = ~bpr_trained_mask[test_target_idx]

print('TEST users:', NUM_TEST)
print('TEST targets unseen in TRAIN+VALID:', int(cold_test_targets.sum()))


## 12. Ranking helpers


In [ ]:
def topk_sorted(scores, k):
    local = np.argpartition(scores, -k, axis=1)[:, -k:]
    top_scores = np.take_along_axis(scores, local, axis=1)
    order = np.argsort(top_scores, axis=1)[:, ::-1]
    local = np.take_along_axis(local, order, axis=1)
    top_scores = np.take_along_axis(top_scores, order, axis=1)
    return local, top_scores


def safe_zscore_1d(values):
    values = np.asarray(values, dtype=np.float32)
    finite = np.isfinite(values)
    out = np.full(values.shape, np.nan, dtype=np.float32)

    if finite.sum() == 0:
        return out
    if finite.sum() == 1:
        out[finite] = 0.0
        return out

    mean = values[finite].mean()
    std = values[finite].std()
    if std < 1e-8:
        out[finite] = 0.0
    else:
        out[finite] = (values[finite] - mean) / std
    return out


def metrics_from_top10(top10, target_idx):
    match = np.where(top10 == int(target_idx))[0]
    if len(match) == 0:
        return 0, 0.0, 0.0
    rank = int(match[0]) + 1
    return 1, 1.0 / rank, 1.0 / np.log2(rank + 1)


## 13. ONE-TIME TEST evaluation

This is the final evaluation. Do not alter `K`, `rho`, recency window, fusion weights, losses, or training settings based on these results.


In [ ]:
bpr_catalog_embeddings = bpr_item_embeddings[final_bpr_catalog_idx]

# Candidate diagnostics
bpr_hits_500 = 0
meta_hits_500 = 0
union_hits = 0
meta_rescued = 0
both_retrieve = 0
neither_retrieve = 0

# BPR top-10 baseline
bpr_hr10 = 0
bpr_mrr10 = 0.0
bpr_ndcg10 = 0.0

# Metadata-only top-10 diagnostic
meta_hr10 = 0
meta_mrr10 = 0.0
meta_ndcg10 = 0.0

# Final hybrid top-10
hybrid_hr10 = 0
hybrid_mrr10 = 0.0
hybrid_ndcg10 = 0.0

union_sizes = []

for start in tqdm(
    range(0, NUM_TEST, EVAL_BATCH_SIZE),
    desc='FINAL TEST evaluation',
):
    end = min(start + EVAL_BATCH_SIZE, NUM_TEST)
    batch_users = test_user_idx[start:end]
    batch_targets = test_target_idx[start:end]

    # BPR scores on final TRAIN+VALID-observed catalog.
    bpr_scores = (
        bpr_user_embeddings[batch_users]
        @ bpr_catalog_embeddings.T
    )

    # Metadata scores on full catalog.
    all_meta_scores = (
        all_history_profiles[batch_users]
        @ metadata_catalog_embeddings.T
    )
    recent_meta_scores = (
        recent_profiles[batch_users]
        @ metadata_catalog_embeddings.T
    )

    # Mask every item seen in TRAIN+VALID.
    for row, user_idx in enumerate(batch_users):
        seen = final_user_seen.get(int(user_idx), set())
        if not seen:
            continue

        seen_global = np.fromiter(seen, dtype=np.int32)

        bpr_pos = global_to_bpr_pos[seen_global]
        bpr_pos = bpr_pos[bpr_pos >= 0]
        if len(bpr_pos):
            bpr_scores[row, bpr_pos] = -np.inf

        all_meta_scores[row, seen_global] = -np.inf
        recent_meta_scores[row, seen_global] = -np.inf

    meta_scores = (
        (1.0 - RHO) * all_meta_scores
        + RHO * recent_meta_scores
    )

    # Frozen top-500 retrieval from each branch.
    bpr_local, _ = topk_sorted(bpr_scores, CANDIDATE_K)
    bpr_candidates = final_bpr_catalog_idx[bpr_local]

    meta_local, _ = topk_sorted(meta_scores, CANDIDATE_K)
    meta_candidates = meta_local.astype(np.int32)

    for r, (user_idx, target_idx) in enumerate(zip(batch_users, batch_targets)):
        bpr_list = bpr_candidates[r]
        meta_list = meta_candidates[r]

        bpr_hit500 = bool(np.any(bpr_list == int(target_idx)))
        meta_hit500 = bool(np.any(meta_list == int(target_idx)))

        bpr_hits_500 += int(bpr_hit500)
        meta_hits_500 += int(meta_hit500)
        union_hits += int(bpr_hit500 or meta_hit500)
        meta_rescued += int((not bpr_hit500) and meta_hit500)
        both_retrieve += int(bpr_hit500 and meta_hit500)
        neither_retrieve += int((not bpr_hit500) and (not meta_hit500))

        # Individual branch top-10 diagnostics.
        h, rr, dcg = metrics_from_top10(bpr_list[:10], target_idx)
        bpr_hr10 += h
        bpr_mrr10 += rr
        bpr_ndcg10 += dcg

        h, rr, dcg = metrics_from_top10(meta_list[:10], target_idx)
        meta_hr10 += h
        meta_mrr10 += rr
        meta_ndcg10 += dcg

        # Union and final frozen 50/50 reranking.
        union = np.unique(np.concatenate([bpr_list, meta_list])).astype(np.int32)
        union_sizes.append(len(union))

        bpr_union_scores = np.full(len(union), np.nan, dtype=np.float32)
        trained = bpr_trained_mask[union]
        if trained.any():
            positions = global_to_bpr_pos[union[trained]]
            bpr_union_scores[trained] = bpr_scores[r, positions]

        meta_union_scores = meta_scores[r, union].astype(np.float32)

        bpr_z = safe_zscore_1d(bpr_union_scores)
        meta_z = safe_zscore_1d(meta_union_scores)

        finite_bpr = np.isfinite(bpr_z)
        if finite_bpr.any():
            bpr_fill = float(bpr_z[finite_bpr].min()) - 1.0
        else:
            bpr_fill = 0.0
        bpr_z = np.where(finite_bpr, bpr_z, bpr_fill)
        meta_z = np.nan_to_num(meta_z, nan=0.0)

        final_scores = ALPHA_BPR * bpr_z + ALPHA_META * meta_z
        top10_local = np.argsort(final_scores)[-10:][::-1]
        final_top10 = union[top10_local]

        h, rr, dcg = metrics_from_top10(final_top10, target_idx)
        hybrid_hr10 += h
        hybrid_mrr10 += rr
        hybrid_ndcg10 += dcg


## 14. Final TEST results


In [ ]:
results = {
    'BPR Recall@500': bpr_hits_500 / NUM_TEST,
    'Metadata Recall@500': meta_hits_500 / NUM_TEST,
    'Union Candidate Recall (500+500)': union_hits / NUM_TEST,
    'Metadata-rescued targets': int(meta_rescued),
    'Both retrieve target': int(both_retrieve),
    'Neither retrieves target': int(neither_retrieve),
    'Mean union size': float(np.mean(union_sizes)),
    'BPR HR@10': bpr_hr10 / NUM_TEST,
    'BPR MRR@10': bpr_mrr10 / NUM_TEST,
    'BPR NDCG@10': bpr_ndcg10 / NUM_TEST,
    'Metadata HR@10': meta_hr10 / NUM_TEST,
    'Metadata MRR@10': meta_mrr10 / NUM_TEST,
    'Metadata NDCG@10': meta_ndcg10 / NUM_TEST,
    'Final Hybrid HR@10': hybrid_hr10 / NUM_TEST,
    'Final Hybrid MRR@10': hybrid_mrr10 / NUM_TEST,
    'Final Hybrid NDCG@10': hybrid_ndcg10 / NUM_TEST,
}

summary = pd.DataFrame([
    {
        'Model': 'Final-refit BPR',
        'Candidate Recall': results['BPR Recall@500'],
        'HR@10': results['BPR HR@10'],
        'MRR@10': results['BPR MRR@10'],
        'NDCG@10': results['BPR NDCG@10'],
    },
    {
        'Model': 'Final-refit Fine-tuned Recency Metadata',
        'Candidate Recall': results['Metadata Recall@500'],
        'HR@10': results['Metadata HR@10'],
        'MRR@10': results['Metadata MRR@10'],
        'NDCG@10': results['Metadata NDCG@10'],
    },
    {
        'Model': 'FINAL HYBRID TEST',
        'Candidate Recall': results['Union Candidate Recall (500+500)'],
        'HR@10': results['Final Hybrid HR@10'],
        'MRR@10': results['Final Hybrid MRR@10'],
        'NDCG@10': results['Final Hybrid NDCG@10'],
    },
])

summary


## 15. Improvement over the final-refit BPR baseline


In [ ]:
relative_improvement = pd.DataFrame([
    {
        'Metric': 'Candidate Recall',
        'Relative improvement vs BPR (%)': (
            results['Union Candidate Recall (500+500)'] / results['BPR Recall@500'] - 1.0
        ) * 100.0,
    },
    {
        'Metric': 'HR@10',
        'Relative improvement vs BPR (%)': (
            results['Final Hybrid HR@10'] / results['BPR HR@10'] - 1.0
        ) * 100.0,
    },
    {
        'Metric': 'MRR@10',
        'Relative improvement vs BPR (%)': (
            results['Final Hybrid MRR@10'] / results['BPR MRR@10'] - 1.0
        ) * 100.0,
    },
    {
        'Metric': 'NDCG@10',
        'Relative improvement vs BPR (%)': (
            results['Final Hybrid NDCG@10'] / results['BPR NDCG@10'] - 1.0
        ) * 100.0,
    },
])

relative_improvement


## 16. Save the final TEST result and frozen artifacts


In [ ]:
payload = {
    'protocol': 'Final refit on TRAIN+VALID, one-time evaluation on TEST',
    'configuration': {
        'bpr_embedding_dim': BPR_DIM,
        'bpr_epochs': BPR_EPOCHS,
        'bpr_lr': BPR_LR,
        'bpr_weight_decay': BPR_WEIGHT_DECAY,
        'minilm_base_model': BASE_MODEL,
        'minilm_max_train_pairs': MAX_TRAIN_PAIRS,
        'minilm_epochs': NUM_EPOCHS,
        'minilm_lr': LEARNING_RATE,
        'candidate_k_per_retriever': CANDIDATE_K,
        'recent_n': RECENT_N,
        'rho': RHO,
        'bpr_weight': ALPHA_BPR,
        'metadata_weight': ALPHA_META,
    },
    'metrics': {
        k: float(v) if isinstance(v, (np.floating, float))
        else int(v) if isinstance(v, (np.integer, int))
        else v
        for k, v in results.items()
    },
    'relative_improvement_vs_final_bpr_pct': {
        row['Metric']: float(row['Relative improvement vs BPR (%)'])
        for _, row in relative_improvement.iterrows()
    },
    'saved_artifacts': {
        'bpr_checkpoint': str(FINAL_BPR_PATH),
        'finetuned_minilm_dir': str(FINAL_MINILM_DIR),
        'finetuned_metadata_embeddings': str(FINAL_META_EMBED_PATH),
    },
}

with open(FINAL_RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2)

print('Saved final TEST results:', FINAL_RESULTS_PATH)
print('Final evaluation complete. Do not tune on TEST.')


## Final frozen architecture

`BPR top-500 + contrastively fine-tuned MiniLM metadata top-500 -> union -> recency-aware metadata scoring (recent-2, rho=0.60) -> 50/50 per-user normalized linear reranking -> top-10`

The TEST result produced above is the final reported result for this experiment.
